# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [1]:
import os
import sys

import pandas as pd

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'raw_transcripts')
transcripts = os.listdir(yt_data_path)

In [16]:
dfs = []

for transcript in tqdm(transcripts):
    transcript_path = os.path.join(yt_data_path, transcript)
    df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
    dfs.append(df)

raw_transcripts_df = DataProcessing.concat_dfs(dfs)
raw_transcripts_df

100%|██████████| 7/7 [00:00<00:00, 191.73it/s]


,Text,Start Time,Duration,Video ID
0,we all know March Madness is all about,0.199,3.081,-rjnvL9LL3U
1,filling out and following your bracket,1.760,3.640,-rjnvL9LL3U
2,so we offer some first round advice,3.280,4.720,-rjnvL9LL3U
3,since you NBC beat Virginia in 2018 the,5.400,5.040,-rjnvL9LL3U
4,16 seed has won two of 24 meetings,8.000,4.960,-rjnvL9LL3U
...,...,...,...,...
2110,and we appreciate,1059.919,1.281,ZZN7BAYeOtc
2111,[cheering and applause] everybody for,1060.457,3.543,ZZN7BAYeOtc
2112,tuning in. Enjoy Super Bowl 60 Sports,1061.200,4.320,ZZN7BAYeOtc
2113,Center with Matt and Hannah's next.,1064.000,4.520,ZZN7BAYeOtc


In [42]:
raw_transcripts_df.head(10)

,Text,Start Time,Duration,Video ID
0,we all know March Madness is all about,0.199,3.081,-rjnvL9LL3U
1,filling out and following your bracket,1.760,3.640,-rjnvL9LL3U
2,so we offer some first round advice,3.280,4.720,-rjnvL9LL3U
3,since you NBC beat Virginia in 2018 the,5.400,5.040,-rjnvL9LL3U
4,16 seed has won two of 24 meetings,8.000,4.960,-rjnvL9LL3U
5,versus the one occasional upset aside,10.440,4.560,-rjnvL9LL3U
6,though the analytics say Advance all the,12.960,4.159,-rjnvL9LL3U
7,one two and three seeds in your bracket,15.000,4.039,-rjnvL9LL3U
8,looking for a long shot the 11 seed has,17.119,3.881,-rjnvL9LL3U
9,won half its meetings with the six seed,19.039,4.801,-rjnvL9LL3U


In [33]:
raw_transcripts_filter_df = raw_transcripts_df.tail(30)
raw_transcripts_filter_df

,Text,Start Time,Duration,Video ID
2085,>> We are in danger of the NFL live curse,1010.240,6.560,ZZN7BAYeOtc
2086,"affecting Super Bowl 60, but it's not",1014.800,4.159,ZZN7BAYeOtc
2087,going to happen. Go with the Patriots.,1016.800,3.120,ZZN7BAYeOtc
2088,">> YES, BOOGIE.",1018.959,3.521,ZZN7BAYeOtc
2089,">> YOU GUYS LOVE ME. I need to. All right,",1019.920,4.720,ZZN7BAYeOtc
2090,but here's why. I actually think Drake,1022.480,3.680,ZZN7BAYeOtc
2091,May and his legs is going to be the key,1024.640,3.120,ZZN7BAYeOtc
2092,factor in this game. I think he's going,1026.160,4.320,ZZN7BAYeOtc
2093,to step up in huge moments.,1027.760,4.400,ZZN7BAYeOtc
2094,">> Yeah. And I'm doing it for you, Mina.",1030.480,4.000,ZZN7BAYeOtc


In [34]:
def clean_data(df) -> list[str]:
    """Rows can contain multiple sentences and some sentences can be on multiple rows, so ensure we 
    join proper sentences together.
    """

    text = df.Text.to_list()
    text_joined = ' '.join(text)
    # print(f"{text_joined}")
    text_split = text_joined.split('.')
    return text_split

In [35]:
cleaned_transcripts = clean_data(raw_transcripts_filter_df)
cleaned_transcripts

[">> We are in danger of the NFL live curse affecting Super Bowl 60, but it's not going to happen",
 ' Go with the Patriots',
 ' >> YES, BOOGIE',
 ' >> YOU GUYS LOVE ME',
 ' I need to',
 " All right, but here's why",
 ' I actually think Drake May and his legs is going to be the key factor in this game',
 " I think he's going to step up in huge moments",
 ' >> Yeah',
 " And I'm doing it for you, Mina",
 " I'm doing it for you",
 " But seriously, Drake May's going to stand up and it's going to be great and they're going to win and you guys are all going to be wrong",
 ' And >> on Sunday night when we see you as part of the Handoff Show, which tune in around 1:00 a',
 'm',
 " Eastern time, we'll talk about how I'm right and you guys are wrong",
 " And then we'll see you again on Monday in Disneyland",
 " We've got all kinds of great coverage still coming up",
 ' Thanks to everybody for being here',
 ' Thanks to our incredible staff back in Bristol, Connecticut',
 ' We love you guys and we

## Prompt LLM

### Create Prompt

In [36]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [37]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'granite-3.3-8b-instruct'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x27c2d6b4890>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x27c29ec6ad0>}

### Pass data + prompt into each LLM

In [38]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: >> We are in danger of the NFL live curse affecting Super Bowl 60, but it's not going to happen
 Model: llama-3.1-8b-instant | Label: 1
 Model: llama-3.3-70b-versatile | Label: 1
1 --- Sentence:  Go with the Patriots
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 1
2 --- Sentence:  >> YES, BOOGIE
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
3 --- Sentence:  >> YOU GUYS LOVE ME
4 --- Sentence:  I need to
5 --- Sentence:  All right, but here's why
6 --- Sentence:  I actually think Drake May and his legs is going to be the key factor in this game
7 --- Sentence:  I think he's going to step up in huge moments
8 --- Sentence:  >> Yeah
9 --- Sentence:  And I'm doing it for you, Mina
10 --- Sentence:  I'm doing it for you
11 --- Sentence:  But seriously, Drake May's going to stand up and it's going to be great and they're going to win and you guys are all going to be wrong
12 --- Sentence:  And >> on Sunda

> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [39]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
40,Enjoy Super Bowl 60 Sports Center with Matt and Hannah's next,"{""label"": 0}",0,None,llama-3.1-8b-instant
41,Enjoy Super Bowl 60 Sports Center with Matt and Hannah's next,"{""label"": 0}",0,None,llama-3.3-70b-versatile
42,>> Amen,"{""label"": 0}",0,None,llama-3.1-8b-instant
43,>> Amen,"{""label"": 0}",0,None,llama-3.3-70b-versatile
44,,"{""label"": 1}",1,None,llama-3.1-8b-instant
45,,"{""label"": 0}",0,None,llama-3.3-70b-versatile


In [40]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile
0,,1,0
1,>> Amen,0,0
2,">> YES, BOOGIE",0,0
3,>> YOU GUYS LOVE ME,0,0
4,>> Yeah,0,0
5,"All right, but here's why",0,0
6,"And >> on Sunday night when we see you as part of the Handoff Show, which tune in around 1:00 a",0,0
7,"And I'm doing it for you, Mina",0,0
8,And then we'll see you again on Monday in Disneyland,0,1
9,"But seriously, Drake May's going to stand up and it's going to be great and they're going to win and you guys are all going to be wrong",1,1


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [41]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,Majority Vote Label
0,,1,0,0
1,>> Amen,0,0,0
2,">> YES, BOOGIE",0,0,0
3,>> YOU GUYS LOVE ME,0,0,0
4,>> Yeah,0,0,0
5,"All right, but here's why",0,0,0
6,"And >> on Sunday night when we see you as part of the Handoff Show, which tune in around 1:00 a",0,0,0
7,"And I'm doing it for you, Mina",0,0,0
8,And then we'll see you again on Monday in Disneyland,0,1,0
9,"But seriously, Drake May's going to stand up and it's going to be great and they're going to win and you guys are all going to be wrong",1,1,1
